# Task 2 - Chunking Strategy Challenge: Semantic Chunking

**Assignment:** Research and implement at least one *alternative* chunking strategy, then **prove** it retrieves a chunk the original method missed.

**Original method:** `RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)` - cuts text at fixed character lengths.

**New method:** `SemanticChunker` (from `langchain_experimental`) - embeds every sentence and starts a **new chunk only when the *meaning* between consecutive sentences shifts** past a threshold. It keeps a full coherent topic together instead of slicing at an arbitrary 500-character boundary.

**Where semantic should win:** the internal SLA policy (Section 1) is one coherent block **longer than 500 characters**, so the original splitter is forced to cut it in half - the *dispatch* rule (noise margin < 6dB) lands in a different chunk than the *compensation* rule (5GB only if outage > 72h). A query needing **both** facts never lands on a single chunk with the whole policy. Semantic chunking keeps it intact.

> Runs on **Google Colab** - the first two cells install the libraries and upload the knowledge-base file.

In [1]:
# === Cell 0: install libraries (Colab) ===
%pip install -q -U \
    langchain langchain-community langchain-core \
    langchain-huggingface langchain-text-splitters langchain-experimental \
    sentence-transformers faiss-cpu
print("\nLibraries installed. If Colab shows a 'RESTART RUNTIME' button, click it, then run from this cell again.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.0/147.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 565.9/565.9 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.3/611.3 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.9/248.9 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests

In [2]:
# === Cell 1: get the knowledge-base file ===
# Upload data/Telecom_Internal_KB.txt from your computer when prompted.
import os
KB_PATH = "Telecom_Internal_KB.txt"
if not os.path.exists(KB_PATH):
    try:
        from google.colab import files
        print("Please choose your Telecom_Internal_KB.txt file to upload...")
        uploaded = files.upload()
        KB_PATH = list(uploaded.keys())[0]
    except Exception:
        raise FileNotFoundError(
            "KB file not found. Upload Telecom_Internal_KB.txt (from the repo's data/ folder) "
            "or set KB_PATH manually."
        )
print(f"Using KB file: {KB_PATH}  ({os.path.getsize(KB_PATH)} bytes)")

Please choose your Telecom_Internal_KB.txt file to upload...


Saving Telecom_Internal_KB.txt to Telecom_Internal_KB.txt
Using KB file: Telecom_Internal_KB.txt  (149010 bytes)


In [3]:
# === Cell 2: imports + embeddings ===
import re, time
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.vectorstores import FAISS

print("Loading embedding model (downloads once on Colab, ~30s)...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)
print("Embeddings ready.")

/tmp/ipykernel_806/4253910858.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
/tmp/ipykernel_806/4253910858.py:8: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


Loading embedding model (downloads once on Colab, ~30s)...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings ready.


## 1. Baseline - the ORIGINAL recursive splitter (size=500, overlap=100)

In [4]:
with open(KB_PATH, encoding="utf-8") as f:
    kb_text = f.read()
documents = TextLoader(KB_PATH, encoding="utf-8").load()

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, chunk_overlap=100,
    length_function=len, separators=["\n\n", "\n", " ", ""]
)
recursive_chunks = recursive_splitter.split_documents(documents)
print(f"ORIGINAL recursive chunks: {len(recursive_chunks)}")

recursive_vs = FAISS.from_documents(recursive_chunks, embeddings)
print("Original FAISS index built.")

ORIGINAL recursive chunks: 481
Original FAISS index built.


## 2. NEW - Semantic Chunking

`SemanticChunker` splits on **meaning shifts**, not character counts. We print the chunk count + size distribution, then show the single chunk that captured the whole SLA policy (the one the original splitter had to cut in half).

In [5]:
t0 = time.time()
print("Building semantic chunks (embeds every sentence - takes ~1 min)...")
semantic_splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"   # new chunk when distance > 95th percentile
)
semantic_chunks = semantic_splitter.create_documents([kb_text])

sizes = [len(c.page_content) for c in semantic_chunks]
print(f"SEMANTIC chunks: {len(semantic_chunks)}  "
      f"(min={min(sizes)}, max={max(sizes)}, avg={sum(sizes)//len(sizes)} chars)")
print(f" built in {time.time()-t0:.0f}s")

for c in semantic_chunks:
    if "6dB" in c.page_content and "72 hours" in c.page_content:
        print("\nSemantic chunk that kept the WHOLE SLA policy together:\n")
        print(c.page_content)
        break

Building semantic chunks (embeds every sentence - takes ~1 min)...
SEMANTIC chunks: 82  (min=204, max=36091, avg=1779 chars)
 built in 101s

Semantic chunk that kept the WHOLE SLA policy together:

# Telecom Egypt Internal Technical Support Knowledge Base (Confidential)

## 1. General Service Level Agreement (SLA) & Dispatch Policies
If a customer reports an internet outage (DSL blinking or no sync):
- The L1 agent must first ensure the customer has restarted the router and checked internal wiring. - If the issue persists for more than 24 hours, the L1 agent must escalate to the Central Exchange team. - A Field Technician must be dispatched if the line noise margin is below 6dB or if the DSL light is completely off/blinking for 3 consecutive days. - The customer must be informed that the technician will contact them within 48 working hours. - Compensation of 5GB mobile data is authorized ONLY if the outage exceeds 72 hours. ## 2. Hardware Specifications & Router Guides

### Router Mode

In [6]:
semantic_vs = FAISS.from_documents(semantic_chunks, embeddings)
print(f"Semantic FAISS index built ({len(semantic_chunks)} vectors).")

Semantic FAISS index built (82 vectors).


## 3. THE PROOF - a chunk the original method missed

**Needle query:** *"What are the conditions for dispatching a field technician, and when is data compensation authorized?"*

The answer needs **two** facts that both live in the SLA policy:
- dispatch trigger -> noise margin **below 6dB** (or DSL off/blinking 3 days)
- compensation -> **5GB only if the outage exceeds 72 hours**

We check each index's top-k for a **single chunk containing BOTH facts**. The original splitter cut the policy in half -> no single chunk has both -> **miss**. Semantic kept it whole -> **hit**.

In [7]:
def first_full_answer_rank(vs, query, must_have, k=3):
    """Rank (1-based) of the first top-k chunk containing ALL required phrases; None if missed."""
    docs = vs.similarity_search(query, k=k)
    for rank, d in enumerate(docs, 1):
        if all(m.lower() in d.page_content.lower() for m in must_have):
            return rank, docs
    return None, docs

query = "What are the conditions for dispatching a field technician, and when is data compensation authorized?"
must_have = ["6dB", "72 hours"]
K = 3

orig_rank, orig_docs = first_full_answer_rank(recursive_vs, query, must_have, k=K)
sem_rank,  sem_docs  = first_full_answer_rank(semantic_vs,  query, must_have, k=K)

print(f"NEEDLE QUERY: {query}")
print(f"Answer must contain BOTH: {must_have}\n")
print(f"ORIGINAL (recursive 500/100): full-answer chunk in top-{K}? -> "
      f"{'MISSED ' if orig_rank is None else f'rank {orig_rank} '}")
print(f"NEW (semantic):               full-answer chunk in top-{K}? -> "
      f"{'MISSED ' if sem_rank is None else f'rank {sem_rank} '}")

print("\n" + "="*72)
print(f"ORIGINAL top-{K} - note NO single chunk has both facts:")
print("="*72)
for i, d in enumerate(orig_docs, 1):
    has6, has72 = ("6dB" in d.page_content), ("72 hours" in d.page_content)
    print(f"[{i}] 6dB={has6} 72h={has72} | {d.page_content[:150].strip().replace(chr(10),' / ')}")

print("\n" + "="*72)
print(f"SEMANTIC top-{K} - the winning chunk contains BOTH facts:")
print("="*72)
for i, d in enumerate(sem_docs, 1):
    has6, has72 = ("6dB" in d.page_content), ("72 hours" in d.page_content)
    star = "  <<< FULL ANSWER" if (has6 and has72) else ""
    print(f"[{i}] 6dB={has6} 72h={has72} | {d.page_content[:150].strip().replace(chr(10),' / ')}{star}")

NEEDLE QUERY: What are the conditions for dispatching a field technician, and when is data compensation authorized?
Answer must contain BOTH: ['6dB', '72 hours']

ORIGINAL (recursive 500/100): full-answer chunk in top-3? -> MISSED 
NEW (semantic):               full-answer chunk in top-3? -> rank 1 

ORIGINAL top-3 - note NO single chunk has both facts:
[1] 6dB=True 72h=False | ## 1. General Service Level Agreement (SLA) & Dispatch Policies / If a customer reports an internet outage (DSL blinking or no sync): / - The L1 agent mus
[2] 6dB=False 72h=True | - The customer must be informed that the technician will contact them within 48 working hours. / - Compensation of 5GB mobile data is authorized ONLY if
[3] 6dB=False 72h=False | #### Error Code E-135 / **Description:** Authentication Timeout / **Resolution Protocol:** Dispatch Field Technician /  / #### Error Code E-136 / **Description:

SEMANTIC top-3 - the winning chunk contains BOTH facts:
[1] 6dB=True 72h=True | # Telecom Egypt In

In [8]:
# (Optional) Broader scan: several policy questions + one error-code lookup.
# Honesty check: semantic does NOT help pinpoint a single error code (near-identical entries get globbed);
# MarkdownHeaderTextSplitter is better for that. We show both outcomes.
candidates = [
    ("conditions to dispatch a technician and when compensation is given", ["6dB", "72 hours"]),
    ("how long before escalating an outage and who to escalate to",        ["24 hours", "Central Exchange"]),
    ("where are fiber optic cuts escalated and what is the SLA",            ["Tier 3 Fiber Ops", "12 hours"]),
    ("resolution protocol for network error code E-317",                   ["E-317"]),
]
print(f"{'ORIG':>6} {'SEM':>6}  query")
for q, must in candidates:
    o, _ = first_full_answer_rank(recursive_vs, q, must, k=3)
    s, _ = first_full_answer_rank(semantic_vs,  q, must, k=3)
    tag = "  <<< semantic wins" if (s is not None and o is None) else ""
    print(f"{str(o):>6} {str(s):>6}  {q}{tag}")

  ORIG    SEM  query
  None      1  conditions to dispatch a technician and when compensation is given  <<< semantic wins
     2      2  how long before escalating an outage and who to escalate to
     1      1  where are fiber optic cuts escalated and what is the SLA
  None   None  resolution protocol for network error code E-317


## 3b. End-to-end proof - same customer inquiry through both chains

The retrieval proof above shows semantic returns a single chunk with the whole policy. Here we show the **practical impact on the LLM's answer**: we feed the **single best chunk** (`k=1`) from each index into the same prompt + Gemini and compare the customer-facing replies.

- **Customer inquiry:** internet down 4 days, DSL light off -> *will you send a technician, and do I get compensation?* (needs BOTH the dispatch rule **and** the compensation rule)
- **Original (recursive):** its best chunk is the *fragment* with the dispatch rule only -> the LLM can mention the technician but **cannot mention compensation** (it isn't in the chunk).
- **Semantic:** its best chunk is the whole SLA policy -> the LLM answers **completely** (technician **and** 5GB / 72h compensation).

> We use `k=1` deliberately to isolate chunk quality. With large `k` the original would eventually pull the second fragment too; the point is that semantic delivers the **complete answer in its very first chunk**.

In [ ]:
# === Install the LLM client + set your Google API key (Colab) ===
%pip install -q -U langchain-google-genai
import os
from getpass import getpass
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
print("Google API key set.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.3/73.3 kB 2.6 MB/s eta 0:00:00
Google API key set.


In [11]:
# === End-to-end proof: SAME original template (from 01_telecom_rag_demo), two chunking strategies ===
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# --- The ORIGINAL system prompt template, copied verbatim from notebook 01 ---
template = """
أنت موظف خدمة عملاء في مزود خدمة إنترنت (ISP).
مهمتك هي الرد على شكوى العميل باللغة العامية المصرية بطريقة مهذبة واحترافية.
تحذير هام: إياك أن تذكر أي اسم شركة اتصالات حقيقي (مثل اتصالات، فودافون، وي، إلخ) في ردك. قدم نفسك فقط كموظف خدمة عملاء فقط.
يجب عليك استخدام المعلومات الموجودة في (السياق الداخلي) فقط لحل المشكلة.
إذا كانت المشكلة تستدعي إرسال فني حسب القواعد، أخبر العميل بذلك بناءً على السياق.
السياق الداخلي (قوانين الشركة وخطوات الحل):
{context}
شكوى العميل:
{question}
الرد:
"""
prompt = PromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

def make_chain(vs, k):
    retriever = vs.as_retriever(search_kwargs={"k": k})
    return ({"context": retriever | format_docs, "question": RunnablePassthrough()}
            | prompt | llm | StrOutputParser())

K = 1  # give the LLM ONLY its single best chunk -> isolates the effect of chunk quality
inquiry = "النت مقطوع عندي من 4 أيام ولمبة الـ DSL طافية خالص. هتبعتولي فني؟ وهل ليا أي تعويض؟"

print(f"CUSTOMER INQUIRY:\n{inquiry}\n")
for name, vs in [("ORIGINAL - RecursiveCharacterTextSplitter (500/100)", recursive_vs),
                 ("NEW - SemanticChunker", semantic_vs)]:
    print("=" * 72)
    print(f"{name} - single top-{K} chunk handed to the LLM:")
    print("=" * 72)
    ctx = vs.similarity_search(inquiry, k=K)
    for d in ctx:
        has_disp = ("6dB" in d.page_content) or ("3 consecutive days" in d.page_content)
        has_comp = ("72 hours" in d.page_content) or ("5GB" in d.page_content)
        print(f"- context chunk (dispatch_rule={has_disp}, compensation_rule={has_comp}):")
        print("   " + d.page_content[:220].replace(chr(10), " / "))
    answer = make_chain(vs, K).invoke(inquiry)
    print(f"\n LLM ANSWER - {name}:\n{answer}\n")

CUSTOMER INQUIRY:
النت مقطوع عندي من 4 أيام ولمبة الـ DSL طافية خالص. هتبعتولي فني؟ وهل ليا أي تعويض؟

ORIGINAL - RecursiveCharacterTextSplitter (500/100) - single top-1 chunk handed to the LLM:
- context chunk (dispatch_rule=True, compensation_rule=False):
   ## 1. General Service Level Agreement (SLA) & Dispatch Policies / If a customer reports an internet outage (DSL blinking or no sync): / - The L1 agent must first ensure the customer has restarted the router and checked inter

 LLM ANSWER - ORIGINAL - RecursiveCharacterTextSplitter (500/100):
أهلاً بحضرتك، أنا موظف خدمة عملاء من مزود خدمة الإنترنت بتاعك.

متأسف جداً على المشكلة اللي حضرتك بتواجهها وإن النت مقطوع عندك بقاله 4 أيام ولمبة الـ DSL طافية خالص.

بناءً على اللي حضرتك ذكرته، بما إن لمبة الـ DSL طافية بقالها 4 أيام متواصلة، ده بيستدعي إننا نبعت لحضرتك فني متخصص عشان يكشف على الخط ويحل المشكلة في أقرب وقت ممكن.

بالنسبة لسؤال حضرتك عن التعويض، ده بيتم مراجعته بعد ما المشكلة تتحل تماماً والخدمة ترجع تشتغل بشكل طبيعي.

ياريت 

In [12]:
# === 3c end-to-end: error-code inquiry, Recursive vs Semantic (k=1), SAME original template ===
# Reuses make_chain / prompt / llm defined in cell 3b.
code = "E-205"
inquiry = "بيظهرلي على الشاشة كود الخطأ E-205 وانا بحاول أستخدم النت، أعمل إيه؟"
print(f"CUSTOMER INQUIRY:\n{inquiry}")
print(f"(KB ground truth: {code} = Line Noise Too High -> Escalate to Tier 2 Network Ops)\n")

for name, vs in [("ORIGINAL - RecursiveCharacterTextSplitter (500/100)", recursive_vs),
                 ("NEW - SemanticChunker", semantic_vs)]:
    print("=" * 72)
    print(f"{name} - single top-1 chunk handed to the LLM:")
    print("=" * 72)
    d = vs.similarity_search(inquiry, k=1)[0]
    nc = len(re.findall(r"E-\d+", d.page_content))
    print(f"- chunk = {len(d.page_content)} chars, {nc} error codes, contains {code}? {code in d.page_content}")
    answer = make_chain(vs, 1).invoke(inquiry)
    print(f"\n LLM ANSWER - {name}:\n{answer}\n")

CUSTOMER INQUIRY:
بيظهرلي على الشاشة كود الخطأ E-205 وانا بحاول أستخدم النت، أعمل إيه؟
(KB ground truth: E-205 = Line Noise Too High -> Escalate to Tier 2 Network Ops)

ORIGINAL - RecursiveCharacterTextSplitter (500/100) - single top-1 chunk handed to the LLM:
- chunk = 468 chars, 4 error codes, contains E-205? True

 LLM ANSWER - ORIGINAL - RecursiveCharacterTextSplitter (500/100):
أهلاً بيك يا فندم، أنا موظف خدمة العملاء من مزود خدمة الإنترنت بتاعك.

متفهم جداً إنك بتواجه مشكلة مع كود الخطأ E-205. الكود ده يا فندم بيشير لمشكلة في جودة الخط، وده بيتطلب تدخل من فريق الدعم الفني المتخصص.

علشان كده، هنحتاج نبعت لحضرتك فني متخصص عشان يفحص الخط ويحل المشكلة من جذورها.

ممكن بعد إذنك تأكدلي رقم حسابك أو رقم التليفون الأرضي المسجل عندنا عشان نقدر نفتح طلب صيانة ونحدد ميعاد مناسب لزيارة الفني؟

متشكرين جداً لتعاونك، وإن شاء الله المشكلة تتحل في أقرب وقت.

NEW - SemanticChunker - single top-1 chunk handed to the LLM:
- chunk = 2196 chars, 1 error codes, contains E-205? False

 LLM ANSWER - NE

## 4. Observations & Proof

**Chunk counts:** Original recursive (500/100) = **481** chunks - Semantic (percentile) = **82** chunks.

**Needle query:** *"What are the conditions for dispatching a field technician, and when is data compensation authorized?"* - requires **both** `6dB` (dispatch) and `72 hours` (compensation), which both live in the Section 1 SLA policy.

**Result (top-3 retrieval):**

| | Original (Recursive 500/100) | New (Semantic) |
|---|---|---|
| Full-answer chunk in top-3? | **MISSED** | **HIT at rank 1** |
| top-1 fact coverage | `6dB=True, 72h=False` | `6dB=True, 72h=True` (one 1007-char chunk) |

- **Why the original misses (deterministic):** across all 481 recursive chunks, **none** contains both facts - the policy was cut in two (chunk 1 = dispatch/`6dB`, chunk 2 = compensation/`72 hours`). Retrieval therefore can never return the whole policy in one chunk.
- **Why semantic wins:** it split on *meaning*, keeping the entire SLA policy in a single chunk -> retrieved complete at rank 1.

**Scan (honesty check):**

| Query | Orig | Sem | |
|---|---|---|---|
| dispatch + compensation | MISS | 1 | semantic wins |
| outage escalation timing | 2 | 2 | tie |
| fiber optic cuts SLA | 1 | 1 | tie |
| error code E-317 | MISS | MISS | both miss |

**Conclusion:** Semantic chunking kept the coherent SLA policy intact, so a multi-fact query retrieved the complete answer the original 500-character splitter had fragmented and missed - satisfying the task.

**Trade-off:** semantic chunking does **not** help pinpoint a single specific error code (`E-317`) - the ~300 near-identical error entries get globbed together. For that access pattern, `MarkdownHeaderTextSplitter` (one clean chunk per `#### Error Code E-xxx`) is the better alternative.